# ProbLog probabilistic neuro-symbolic reasoner — demo

**Artifact:** *LLM-as-probabilistic-reasoner over a decoy-gated, FDR-certified knowledge base, with auditable probabilistic trace-graphs.*

This notebook demonstrates the **P4 deliverable** of the pipeline that translates short
legal / news / regulatory documents into gated first-order-logic facts and reasons over them:

1. The **deterministic backward-chaining engine** (`kb_engine.py`) — a pure-Python vanilla
   meta-interpreter over admitted facts + hand-authored genre bridge rules, exporting
   human-auditable trace-graphs.
2. The **probabilistic reasoner** (`prob_reasoner.py`) — every admitted leaf carries an
   LLM-supplied, FDR-certificate-consistent unification **weight**; each document's KB is
   compiled to a **weighted ProbLog program** and multi-hop conclusion **marginals** are
   computed via `get_evaluatable().create_from(PrologString(prog)).evaluate()`, with a
   pure-Python **exact weighted-model-counting** fallback validated equal to ProbLog to 1e-9.

The demo runs entirely on **CPU**, makes **no LLM calls** (it is a `$0` cache-hit reanalysis),
and:

* runs both modules' self-tests (toy 2-hop marginal = 0.63; ProbLog == exact-WMC, including
  the shared-leaf case where naive noisy-OR fails),
* **re-evaluates the stored weighted ProbLog programs** for real legal/news/regulatory
  documents and checks the recomputed marginals reproduce the published values,
* shows the certificate→weight **sensitivity** table (shrinkage ≤ identity everywhere),
* reports the **honest finalized** metrics (atomic-fact reduction, multi-hop corruption,
  conservative self-report regime vs the CLUTRR anti-conservative contrast),
* and renders the four headline figures + a **probabilistic trace-graph**.

The certificate→weight map (default): `w_i = (1 - alpha_hat) * calibrate(Z_i)`, with
`alpha_hat = decoy_fdr_hat` of the operative gate cell; the per-pair margin
`0.5 + 0.5*W_i` and the identity (no-shrinkage) map are reported as a sensitivity baseline.

In [ ]:
# --- Install dependencies (works on Colab and locally) ---
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# ProbLog — the probabilistic-logic engine (NOT pre-installed on Colab; pure-Python,
# only needs setuptools, so it is safe to install everywhere)
_pip('problog==2.2.10')

# Core scientific / plotting packages — pre-installed on Colab; install locally only,
# at Colab's exact versions, so the local environment matches Colab.
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'networkx==3.6.1', 'matplotlib==3.10.0')

In [ ]:
# --- Notebook-level imports (the two inlined modules bring their own imports) ---
import json, os, math, urllib.request
import matplotlib.pyplot as plt
import networkx as nx

In [ ]:
# --- Load the curated demo data (GitHub URL with local fallback for Colab) ---
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-6db730-decoy-gated-neuro-symbolic-extraction-a/main/round-4/experiment-3/demo/mini_demo_data.json"

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print("loaded keys:", list(data.keys()))
print("curated documents:", [t["doc_id"] for t in data["prob_trace_graphs"]])
print("engine:", data["meta"]["engine"], "| problog_available:", data["meta"]["problog_available"])
print("self-report regime:", data["meta"]["self_report_regime"])

## Configuration

All tunable parameters live here. They start at the **minimum** that still produces
meaningful output; raise them toward the commented original values for a fuller run.
Everything is deterministic and CPU-only, so scaling is cheap.

In [ ]:
# --- Tunable parameters (minimums; original/full values in comments) ---
RUN_SELFTESTS       = True   # run kb_engine + prob_reasoner self-tests (ProbLog == exact-WMC)
MAX_TRACE_DOCS      = 4      # documents whose stored ProbLog programs we re-evaluate.  min: 1   full: 4 (all curated)
MAX_SENSITIVITY_ROWS = 12    # rows of the certificate->weight sensitivity table.        min: 4   full: 12
F4_DOC_ID           = "reg_gdpr_13"   # document whose probabilistic trace-graph we render (F4)

print(f"RUN_SELFTESTS={RUN_SELFTESTS}  MAX_TRACE_DOCS={MAX_TRACE_DOCS}  "
      f"MAX_SENSITIVITY_ROWS={MAX_SENSITIVITY_ROWS}  F4_DOC_ID={F4_DOC_ID!r}")

## Module 1 — `kb_engine.py` (deterministic backward-chaining engine)

The default reasoning engine: a pure-Python vanilla meta-interpreter over admitted facts +
non-recursive, range-restricted genre bridge rules. Each leaf resolves against the
admitted-fact table and carries a **certificate**
`{provenance, decoy_certificate:{W_i,T,alpha}, entrapment_certificate:{FDP_hat,r}}`, so every
derived conclusion is traceable back to gated, provenance-bearing base facts. Proofs flatten
to `{nodes, edges}` graphs that render to Graphviz DOT.

*The cell below is the original `kb_engine.py`, copied verbatim, with one small notebook-only
addition at the end: it registers the inlined definitions as an importable `kb_engine` module
so the next cell's `import kb_engine as kbe` resolves to exactly these objects.*

In [ ]:
#!/usr/bin/env python3
"""
kb_engine.py — pure-Python backward-chaining meta-interpreter over admitted facts +
hand-authored genre bridge rules, with auditable trace-graph export (JSON + Graphviz DOT).

This is the DEFAULT reasoning engine (the plan's deliverable): it satisfies the goal's
"running logic interpreter + human-auditable trace-graphs" requirement without a system
SWI-Prolog. It mirrors the classic vanilla meta-interpreter

    solve(true, true).
    solve((A,B), (PA,PB)) :- solve(A,PA), solve(B,PB).
    solve(A, node(A, Rule, Proof)) :- (A :- Body), solve(Body, Proof).
    solve(A, leaf(A, Cert)) :- admitted_fact(A, Cert).

Each leaf resolves against the admitted-fact table and carries a certificate
    cert = {provenance_char_span, decoy_certificate:{W_i,T,alpha}, entrapment_certificate:{FDP_hat,r}}
so every derived conclusion is traceable to gated, provenance-bearing base facts.

A Fact is a ground atom: (pred, (arg1, arg2, ...)).
A Rule is (head_pred, head_args, body) where head_args/body atoms use string VARIABLES
(identifiers starting with an uppercase letter) and constants (anything else).
Rules here are non-recursive and range-restricted, so SLD resolution terminates.
"""
from __future__ import annotations

import html
import itertools
from pathlib import Path

# ---------------------------------------------------------------------------
# Term helpers — explicit logic variables (entity constants are arbitrary strings,
# so a capitalization convention cannot distinguish vars from constants).
# ---------------------------------------------------------------------------
class Var:
    __slots__ = ("name",)

    def __init__(self, name: str):
        self.name = name

    def __eq__(self, other):
        return isinstance(other, Var) and other.name == self.name

    def __hash__(self):
        return hash(("Var", self.name))

    def __repr__(self):
        return f"?{self.name}"


def V(name: str) -> Var:
    return Var(name)


def is_var(x) -> bool:
    return isinstance(x, Var)


def unify(pat, val, subst: dict) -> dict | None:
    """Unify a (possibly variable-bearing) arg tuple `pat` with ground tuple `val`."""
    s = dict(subst)
    for p, v in zip(pat, val):
        if is_var(p):
            if p in s:
                if s[p] != v:
                    return None
            else:
                s[p] = v
        elif p != v:
            return None
    return s


def subst_args(args, subst: dict) -> tuple:
    return tuple(subst.get(a, a) if is_var(a) else a for a in args)


class KB:
    """Admitted facts + bridge rules + a backward-chaining solver with proof capture."""

    def __init__(self) -> None:
        # (pred, args_tuple) -> certificate dict
        self.facts: dict[tuple, dict] = {}
        self.by_pred: dict[str, list[tuple]] = {}
        self.rules: list[dict] = []

    def add_fact(self, pred: str, args: tuple, cert: dict) -> None:
        key = (pred, tuple(args))
        if key not in self.facts:
            self.facts[key] = cert
            self.by_pred.setdefault(pred, []).append(tuple(args))

    def add_rule(self, name: str, head_pred: str, head_args: tuple, body: list) -> None:
        """body: list of (pred, args) atoms; vars are shared across head+body."""
        self.rules.append({"name": name, "head_pred": head_pred,
                           "head_args": tuple(head_args), "body": list(body)})

    # -- backward chaining ---------------------------------------------------
    @staticmethod
    def _rename(atom_args, tag: str):
        """Rename variables in an arg tuple with a unique tag to avoid clashes."""
        return tuple(Var(f"{a.name}_{tag}") if is_var(a) else a for a in atom_args)

    def _solve_atom(self, pred: str, args: tuple, subst: dict, depth: int):
        """Yield (new_subst, proof_node) for goal pred(args) under subst.

        Goal args are first grounded through subst; remaining unbound vars are matched
        against facts (binding them) or expanded via rules with fresh-renamed variables.
        """
        g_args = subst_args(args, subst)
        # 1) base facts
        for fact_args in self.by_pred.get(pred, []):
            s2 = unify(g_args, fact_args, subst)
            if s2 is not None:
                cert = self.facts[(pred, fact_args)]
                yield s2, {"type": "leaf", "atom": [pred, list(fact_args)], "cert": cert}
        # 2) rule expansion (non-recursive bridges; depth cap is a safety net)
        if depth <= 0:
            return
        for ri, rule in enumerate(self.rules):
            if rule["head_pred"] != pred:
                continue
            tag = f"{depth}_{ri}"
            head = self._rename(rule["head_args"], tag)
            s_head = unify(head, g_args, subst)  # bind renamed head vars to the (ground) goal
            if s_head is None:
                continue
            body = [(p, self._rename(a, tag)) for (p, a) in rule["body"]]
            for sb, child_proofs in self._solve_body(body, s_head, depth - 1):
                head_ground = subst_args(head, sb)
                if any(is_var(a) for a in head_ground):
                    continue
                yield sb, {"type": "derived", "atom": [pred, list(head_ground)],
                           "rule": rule["name"], "children": child_proofs}

    def _solve_body(self, body: list, subst: dict, depth: int):
        if not body:
            yield subst, []
            return
        first, rest = body[0], body[1:]
        for s2, proof in self._solve_atom(first[0], first[1], subst, depth):
            for s3, proofs in self._solve_body(rest, s2, depth):
                yield s3, [proof] + proofs

    def run_rule(self, rule: dict, max_depth: int = 4):
        """Solve a rule's body over the KB and yield fully-ground derived proofs."""
        body = list(rule["body"])
        for sb, child_proofs in self._solve_body(body, {}, max_depth):
            head_ground = subst_args(rule["head_args"], sb)
            if any(is_var(a) for a in head_ground):
                continue
            yield {"type": "derived", "atom": [rule["head_pred"], list(head_ground)],
                   "rule": rule["name"], "children": child_proofs}

    def derive_all(self, max_depth: int = 4) -> list[dict]:
        """Run every rule and collect distinct derived conclusions with one proof each."""
        seen, out = set(), []
        for rule in self.rules:
            for proof in self.run_rule(rule, max_depth):
                key = (proof["atom"][0], tuple(proof["atom"][1]))
                if key in seen:
                    continue
                seen.add(key)
                out.append(proof)
        return out


# ---------------------------------------------------------------------------
# Proof-graph flattening + leaf walk
# ---------------------------------------------------------------------------
def iter_leaves(proof: dict):
    if proof["type"] == "leaf":
        yield proof
    else:
        for c in proof.get("children", []):
            yield from iter_leaves(c)


def proof_to_graph(proof: dict) -> dict:
    """Flatten a proof tree into {nodes:[{id,label,kind,cert?}], edges:[{src,dst,rule}]}."""
    nodes, edges = [], []
    counter = itertools.count()

    def atom_str(atom):
        pred, args = atom[0], atom[1]
        return f"{pred}({', '.join(map(str, args))})"

    def walk(node) -> int:
        nid = next(counter)
        if node["type"] == "leaf":
            nodes.append({"id": nid, "label": atom_str(node["atom"]),
                          "kind": "leaf", "cert": node.get("cert")})
        else:
            nodes.append({"id": nid, "label": atom_str(node["atom"]),
                          "kind": "derived", "rule": node.get("rule")})
            for c in node.get("children", []):
                cid = walk(c)
                edges.append({"src": nid, "dst": cid, "rule": node.get("rule")})
        return nid

    walk(proof)
    return {"nodes": nodes, "edges": edges}


def graph_to_dot(graph: dict, title: str = "") -> str:
    """Render a flattened proof graph to Graphviz DOT.

    Node colour encodes gate status: derived=lightblue, admitted-entailed leaf=palegreen,
    hallucinated leaf=lightsalmon. Leaf tooltip carries provenance + W_i + FDP_hat.
    """
    lines = ["digraph proof {", '  rankdir=TB;', '  node [style=filled, fontname="Helvetica", fontsize=10];']
    if title:
        lines.append(f'  labelloc="t"; label="{html.escape(title)}";')
    for n in graph["nodes"]:
        label = html.escape(n["label"])
        if n["kind"] == "derived":
            color, extra = "lightblue", f'\\nrule: {html.escape(str(n.get("rule")))}'
            tooltip = "derived conclusion"
        else:
            cert = n.get("cert") or {}
            hv = cert.get("hallucination_verdict", "?")
            color = "lightsalmon" if hv == "HALLUCINATED" else "palegreen"
            dc = cert.get("decoy_certificate") or {}
            ec = cert.get("entrapment_certificate") or {}
            extra = (f'\\nW={dc.get("W_i")} T={dc.get("T")} a={dc.get("alpha")}'
                     f'\\nFDP_hat={ec.get("FDP_hat")} r={ec.get("r")}')
            tooltip = html.escape(str(cert.get("provenance", ""))[:200] or "leaf fact")
        lines.append(f'  n{n["id"]} [label="{label}{extra}", fillcolor="{color}", '
                     f'tooltip="{tooltip}"];')
    for e in graph["edges"]:
        lines.append(f'  n{e["src"]} -> n{e["dst"]} [label="{html.escape(str(e.get("rule") or ""))}", '
                     f'fontsize=8];')
    lines.append("}")
    return "\n".join(lines)


def selftest() -> None:
    kb = KB()
    # toy 2-hop derivation: cross_references(a,b), grants_right(b,r) => relevant_right(a,r)
    kb.add_fact("cross_references", ("Art13", "Art6"),
                {"provenance": "Art.13 refers to Art.6", "hallucination_verdict": "ENTAILED",
                 "decoy_certificate": {"W_i": 0.9, "T": 0.4, "alpha": 0.2},
                 "entrapment_certificate": {"FDP_hat": 0.05, "r": 1}})
    kb.add_fact("grants_right", ("Art6", "lawful_processing"),
                {"provenance": "Art.6 grants the right to lawful processing",
                 "hallucination_verdict": "ENTAILED",
                 "decoy_certificate": {"W_i": 0.7, "T": 0.4, "alpha": 0.2},
                 "entrapment_certificate": {"FDP_hat": 0.05, "r": 1}})
    kb.add_rule("relevant_right", "relevant_right", (V("A"), V("R")),
                [("cross_references", (V("A"), V("B"))), ("grants_right", (V("B"), V("R")))])
    derived = kb.derive_all()
    assert len(derived) == 1, f"expected 1 derived, got {len(derived)}"
    d = derived[0]
    assert d["atom"][0] == "relevant_right" and d["atom"][1] == ["Art13", "lawful_processing"], d["atom"]
    leaves = list(iter_leaves(d))
    assert len(leaves) == 2 and all("cert" in lf and lf["cert"].get("decoy_certificate")
                                    and lf["cert"].get("entrapment_certificate")
                                    and "provenance" in lf["cert"] for lf in leaves), \
        "every leaf must carry all three certificate fields"
    g = proof_to_graph(d)
    dot = graph_to_dot(g, title="toy")
    assert dot.startswith("digraph proof {") and "relevant_right" in dot
    assert "->" in dot and "fillcolor" in dot
    print("kb_engine selftest PASSED")


if __name__ == "__main__":
    selftest()


# --- notebook glue: expose the definitions above as an importable `kb_engine` module so
# prob_reasoner.py's `import kb_engine as kbe` / `from kb_engine import V` resolve to them ---
import sys as _sys, types as _types
kb_engine = _types.ModuleType("kb_engine")
for _name in ("Var", "V", "is_var", "unify", "subst_args", "KB", "iter_leaves",
              "proof_to_graph", "graph_to_dot", "selftest"):
    setattr(kb_engine, _name, globals()[_name])
_sys.modules["kb_engine"] = kb_engine

## Module 2 — `prob_reasoner.py` (LLM-as-probabilistic-reasoner)

Turns each document's admitted-fact KB + bridge rules into a **weighted ProbLog program**
and computes every multi-hop conclusion's **marginal** via weighted model counting. Two
interchangeable, validated-equivalent engines:

* **ProbLog** (primary): `get_evaluatable().create_from(PrologString(prog)).evaluate()`.
* **Exact weighted model counting** (pure-Python fallback): enumerate truth assignments of the
  distinct rule-feeding leaves and accumulate assignment probability whenever the conclusion
  is derivable — identical to ProbLog's WMC for independent Bernoulli leaves + monotone rules,
  including the shared-leaf case where naive noisy-OR over-counts. A flagged noisy-OR
  approximation kicks in only above an enumeration cap.

Certificate→weight maps: gate-consistent shrinkage (default), per-pair knockoff margin, and
identity / no-shrinkage (sensitivity baseline).

*The cell below is the original `prob_reasoner.py`, copied verbatim.*

In [ ]:
#!/usr/bin/env python3
"""
prob_reasoner.py — the LLM-as-probabilistic-reasoner layer (P4 deliverable).

This module turns the deterministic backward-chaining KB (kb_engine.KB) of admitted,
provenance- and certificate-bearing facts + hand-authored genre bridge rules into a
PROBABILISTIC program whose every leaf carries an LLM-supplied, FDR-certificate-consistent
unification weight, and computes the MARGINAL probability of every derived multi-hop
conclusion via weighted model counting.

Two interchangeable, *equivalent* engines (validated against each other in selftest):
  * ProbLog (primary): get_evaluatable().create_from(PrologString(prog)).evaluate()  -> {Term:prob}
    exactly per the verified research spec (Part C.5 deterministic->probabilistic swap).
  * Pure-Python EXACT weighted-model-count fallback (if ProbLog cannot install/run on a
    minimal CPU image): enumerate truth assignments of the distinct grounded leaves that
    feed the rules, run the existing deterministic kb.derive_all on each present-subset, and
    accumulate the assignment probability whenever the queried conclusion is derivable.
    For independent Bernoulli leaves + deterministic monotone rules this is IDENTICAL to
    ProbLog's WMC. A noisy-OR proof-level approximation is used only if the relevant-leaf
    count exceeds an enumeration cap (flagged explicitly in the output).

The deterministic engine REMAINS the baseline: NO headline number depends on this module.
Here we only ADD a probabilistic marginal + a probabilistic trace-graph on top of the
already-derived (and already-gated) proofs.

Certificate -> weight mapping (research spec Part C.2):
  p_i = calibrate(Z_i)                          # Z_i = per-doc rank-normalized real score
  (i)  gate-consistent shrinkage  w_i = (1 - alpha_hat) * p_i        [DEFAULT, headline]
  (ii) per-pair margin            w_i = clip(0.5 + 0.5 * W_i, eps, 1-eps)
  (iii) identity / no-shrinkage   w_i = p_i  (alpha_hat = 0)         [sensitivity baseline]
  entrapment FDP_hat is carried at the leaf as a consistency prior (annotation only).
"""
from __future__ import annotations

import re
from collections import defaultdict

import kb_engine as kbe
from kb_engine import V  # noqa: F401  (re-exported for selftest convenience)

EPS = 1e-3
ENUM_CAP = 18  # max distinct rule-feeding leaves for exact enumeration (2^18 = 262144)

# ---------------------------------------------------------------------------
# ProbLog availability (detected once at import; never fatal)
# ---------------------------------------------------------------------------
try:  # pragma: no cover - environment dependent
    from problog.program import PrologString
    from problog import get_evaluatable
    _PROBLOG_OK = True
except Exception:  # pragma: no cover
    PrologString = None
    get_evaluatable = None
    _PROBLOG_OK = False


def problog_available() -> bool:
    return _PROBLOG_OK


# ---------------------------------------------------------------------------
# Calibration + certificate -> weight maps
# ---------------------------------------------------------------------------
def calibrate(z, eps: float = EPS) -> float:
    """Z_i (per-doc rank-normalized real score in [0,1]) -> p_i in (0,1).

    DEFAULT = identity clamp: monotone, label-free (consistent with the label-free gate).
    Missing score -> 0.5 (no information)."""
    if z is None:
        return 0.5
    return min(1.0 - eps, max(eps, float(z)))


def weight_gate_consistent(z, alpha_hat: float, eps: float = EPS) -> float:
    """(i) DEFAULT: gate-consistent shrinkage w_i = (1 - alpha_hat) * calibrate(Z_i)."""
    p = calibrate(z, eps)
    ah = 0.0 if alpha_hat is None else float(alpha_hat)
    ah = min(1.0, max(0.0, ah))
    return min(1.0 - eps, max(eps, (1.0 - ah) * p))


def weight_margin(w_i, eps: float = EPS) -> float:
    """(ii) per-pair knockoff-margin weight w_i = clip(0.5 + 0.5*W_i, eps, 1-eps).

    W_i is the signed-max knockoff statistic built from rank-normalized real/decoy scores,
    so it already lives in [-1, 1]; the affine map sends the antisymmetric margin to a
    probability in (0,1) (0.5 at an exchangeable tie, ->1 as the real dominates its decoy)."""
    if w_i is None:
        return 0.5
    return min(1.0 - eps, max(eps, 0.5 + 0.5 * float(w_i)))


def weight_identity(z, eps: float = EPS) -> float:
    """(iii) identity / no-shrinkage baseline w_i = calibrate(Z_i) (alpha_hat = 0)."""
    return calibrate(z, eps)


# ---------------------------------------------------------------------------
# Prolog-atom sanitisation (functor slug + single-quoted constants) + back-map
# ---------------------------------------------------------------------------
def slug_functor(name: str) -> str:
    """Predicate functor -> valid lowercase Prolog atom: [^a-z0-9_]->'_', 'p_' prefix if
    it does not begin with a lowercase letter."""
    s = re.sub(r"[^a-z0-9_]", "_", str(name).lower())
    if not s or not s[0].isalpha():
        s = "p_" + s
    return s


def quote_const(name) -> str:
    """Entity constant -> single-quoted Prolog atom, inner quotes/backslashes escaped."""
    s = str(name).replace("\\", "\\\\").replace("'", "\\'")
    return "'" + s + "'"


def var_name(name: str) -> str:
    """kb_engine Var name -> valid Prolog variable (must start uppercase / underscore)."""
    s = re.sub(r"[^A-Za-z0-9_]", "_", str(name))
    if not s or not (s[0].isupper() or s[0] == "_"):
        s = "V_" + s
    return s


def _canon(atom_str: str) -> str:
    """Whitespace/quote-insensitive canonical key for matching ProbLog Term strings back
    to the (pred, args) we emitted. ProbLog drops unnecessary quotes in str(Term), so we
    strip quotes + whitespace on both sides before comparing."""
    return atom_str.replace("'", "").replace(" ", "")


def _render_atom(pred: str, args, is_rule: bool) -> str:
    """Render one atom. In a rule body/head, kb_engine Var objects become Prolog variables;
    everywhere else ground constants become single-quoted atoms."""
    parts = []
    for a in args:
        if kbe.is_var(a):
            parts.append(var_name(a.name))
        else:
            parts.append(quote_const(a))
    return f"{slug_functor(pred)}({','.join(parts)})"


def build_program(kb: kbe.KB, leaf_weights: dict, conclusions: list):
    """Assemble a ProbLog program string + a back-map for the queried conclusions.

    leaf_weights : {(pred, args_tuple) -> w_i in (0,1)} for the admitted facts.
    conclusions  : list of (pred, args) atoms to query (derived multi-hop heads).
    Returns (program_str, query_map) where query_map: canonical_atom_str -> (pred, args_tuple).
    """
    lines: list[str] = []
    # weighted facts
    for (pred, args), _cert in kb.facts.items():
        w = leaf_weights.get((pred, tuple(args)), 0.5)
        w = min(1.0 - EPS, max(EPS, float(w)))
        lines.append(f"{w:.6f}::{_render_atom(pred, args, is_rule=False)}.")
    # deterministic bridge rules (weight 1)
    for rule in kb.rules:
        head = _render_atom(rule["head_pred"], rule["head_args"], is_rule=True)
        body = ", ".join(_render_atom(p, a, is_rule=True) for (p, a) in rule["body"])
        lines.append(f"{head} :- {body}.")
    # queries
    query_map: dict[str, tuple] = {}
    seen = set()
    for (pred, args) in conclusions:
        atomstr = _render_atom(pred, args, is_rule=False)
        if atomstr in seen:
            continue
        seen.add(atomstr)
        lines.append(f"query({atomstr}).")
        query_map[_canon(atomstr)] = (pred, tuple(args))
    return "\n".join(lines), query_map


# ---------------------------------------------------------------------------
# Engine 1: ProbLog
# ---------------------------------------------------------------------------
def run_problog(program: str, query_map: dict) -> dict:
    """Evaluate a ProbLog program; map Term marginals back to (pred,args) -> prob.
    Raises on any ProbLog failure so the caller can fall back."""
    if not _PROBLOG_OK:
        raise RuntimeError("problog not importable")
    res = get_evaluatable().create_from(PrologString(program)).evaluate()
    out = {}
    for term, prob in res.items():
        key = query_map.get(_canon(str(term)))
        if key is not None:
            out[key] = float(prob)
    return out


# ---------------------------------------------------------------------------
# Engine 2: pure-Python EXACT weighted model counting (ProbLog-equivalent)
# ---------------------------------------------------------------------------
def _sub_kb(present_leaves, rules) -> kbe.KB:
    kb = kbe.KB()
    for (pred, args) in present_leaves:
        kb.add_fact(pred, args, {})
    for rule in rules:
        kb.add_rule(rule["name"], rule["head_pred"], rule["head_args"], rule["body"])
    return kb


def all_proofs_by_conclusion(kb: kbe.KB, max_depth: int = 4) -> dict:
    """Collect EVERY ground proof per conclusion (not deduped) — needed for noisy-OR."""
    out: dict[tuple, list] = defaultdict(list)
    for rule in kb.rules:
        for proof in kb.run_rule(rule, max_depth=max_depth):
            key = (proof["atom"][0], tuple(proof["atom"][1]))
            out[key].append(proof)
    return out


def run_wmc(kb: kbe.KB, leaf_weights: dict, conclusions: list,
            enum_cap: int = ENUM_CAP) -> tuple[dict, str]:
    """Exact WMC over the distinct rule-feeding leaves (ProbLog-equivalent for independent
    Bernoulli facts + deterministic monotone rules). Falls back to a flagged noisy-OR
    proof approximation only if that leaf count exceeds enum_cap.
    Returns ({(pred,args_tuple) -> marginal}, engine_tag)."""
    concl_keys = [(c[0], tuple(c[1])) for c in conclusions]
    if not concl_keys:
        return {}, ("fallback_exact_wmc" if not _PROBLOG_OK else "fallback_exact_wmc")
    relevant_preds = {p for rule in kb.rules for (p, _a) in rule["body"]}
    rl = [(pred, tuple(args)) for (pred, args) in kb.facts.keys() if pred in relevant_preds]
    n = len(rl)
    if n <= enum_cap:
        ws = [min(1.0 - EPS, max(EPS, float(leaf_weights.get(rl[i], 0.5)))) for i in range(n)]
        marg = {ck: 0.0 for ck in concl_keys}
        for mask in range(1 << n):
            p = 1.0
            for i in range(n):
                p *= ws[i] if (mask & (1 << i)) else (1.0 - ws[i])
            if p == 0.0:
                continue
            present = [rl[i] for i in range(n) if (mask & (1 << i))]
            sub = _sub_kb(present, kb.rules)
            derived = {(pp["atom"][0], tuple(pp["atom"][1]))
                       for pp in sub.derive_all(max_depth=4)}
            for ck in concl_keys:
                if ck in derived:
                    marg[ck] += p
        return {ck: min(1.0, max(0.0, v)) for ck, v in marg.items()}, "fallback_exact_wmc"
    # noisy-OR proof-level approximation (exact only when proofs share no leaves)
    proofs_by = all_proofs_by_conclusion(kb)
    marg = {}
    for ck in concl_keys:
        prod_not = 1.0
        for proof in proofs_by.get(ck, []):
            pl = 1.0
            for lf in kbe.iter_leaves(proof):
                key = (lf["atom"][0], tuple(lf["atom"][1]))
                pl *= min(1.0 - EPS, max(EPS, float(leaf_weights.get(key, 0.5))))
            prod_not *= (1.0 - pl)
        marg[ck] = min(1.0, max(0.0, 1.0 - prod_not))
    return marg, "fallback_noisy_or_approx"


# ---------------------------------------------------------------------------
# Unified marginal computation (try ProbLog, else WMC) — never raises
# ---------------------------------------------------------------------------
def conclusion_marginals(kb: kbe.KB, leaf_weights: dict, conclusions: list,
                         prefer_problog: bool = True) -> tuple[dict, str]:
    """Return ({(pred,args_tuple) -> marginal}, engine in {'problog','fallback_exact_wmc',
    'fallback_noisy_or_approx'}). On any ProbLog error, fall back to exact WMC."""
    if prefer_problog and _PROBLOG_OK:
        try:
            program, qmap = build_program(kb, leaf_weights, conclusions)
            marg = run_problog(program, qmap)
            # ProbLog can silently omit unreachable queries -> backfill via WMC
            missing = [c for c in conclusions if (c[0], tuple(c[1])) not in marg]
            if missing:
                wm, _ = run_wmc(kb, leaf_weights, missing)
                marg.update(wm)
            return marg, "problog"
        except Exception:  # pragma: no cover - parse/compile failure -> WMC
            pass
    return run_wmc(kb, leaf_weights, conclusions)


# ---------------------------------------------------------------------------
# Probabilistic trace-graph (reuse proof shape; add a 'prob' attribute per node)
# ---------------------------------------------------------------------------
def proof_to_prob_graph(proof: dict, leaf_weights: dict, marginals: dict) -> dict:
    """Flatten a proof into {nodes,edges} like kb_engine.proof_to_graph, but with a 'prob'
    field on every node: leaf prob = its calibrated weight w_i; derived/root prob = the
    (ProbLog/WMC) marginal of that sub-conclusion. Leaf certificates are preserved."""
    nodes, edges = [], []
    counter = [0]

    def atom_str(atom):
        return f"{atom[0]}({', '.join(map(str, atom[1]))})"

    def walk(node) -> int:
        nid = counter[0]
        counter[0] += 1
        key = (node["atom"][0], tuple(node["atom"][1]))
        if node["type"] == "leaf":
            w = leaf_weights.get(key)
            w = float(w) if w is not None else None
            nodes.append({"id": nid, "label": atom_str(node["atom"]), "kind": "leaf",
                          "prob": (round(w, 6) if w is not None else None),
                          "cert": node.get("cert")})
        else:
            m = marginals.get(key)
            nodes.append({"id": nid, "label": atom_str(node["atom"]), "kind": "derived",
                          "rule": node.get("rule"),
                          "prob": (round(float(m), 6) if m is not None else None)})
            for c in node.get("children", []):
                cid = walk(c)
                edges.append({"src": nid, "dst": cid, "rule": node.get("rule")})
        return nid

    walk(proof)
    return {"nodes": nodes, "edges": edges}


def prob_graph_to_dot(graph: dict, title: str = "") -> str:
    """DOT rendering with marginal/weight annotated on every node label."""
    import html
    lines = ["digraph prob_proof {", "  rankdir=TB;",
             '  node [style=filled, fontname="Helvetica", fontsize=10];']
    if title:
        lines.append(f'  labelloc="t"; label="{html.escape(title)}";')
    for n in graph["nodes"]:
        label = html.escape(n["label"])
        pr = n.get("prob")
        pr_s = f"\\np={pr:.3f}" if isinstance(pr, (int, float)) else ""
        if n["kind"] == "derived":
            color = "lightblue"
            extra = f'\\nrule: {html.escape(str(n.get("rule")))}{pr_s}'
            tooltip = "derived conclusion (marginal)"
        else:
            cert = n.get("cert") or {}
            hv = cert.get("hallucination_verdict", "?")
            color = "lightsalmon" if hv == "HALLUCINATED" else "palegreen"
            dc = cert.get("decoy_certificate") or {}
            ec = cert.get("entrapment_certificate") or {}
            extra = (f'{pr_s}\\nW={dc.get("W_i")} T={dc.get("T")} a={dc.get("alpha")}'
                     f'\\nFDP_hat={ec.get("FDP_hat")} r={ec.get("r")}')
            tooltip = html.escape(str(cert.get("provenance", ""))[:200] or "leaf fact")
        lines.append(f'  n{n["id"]} [label="{label}{extra}", fillcolor="{color}", '
                     f'tooltip="{tooltip}"];')
    for e in graph["edges"]:
        lines.append(f'  n{e["src"]} -> n{e["dst"]} [label="{html.escape(str(e.get("rule") or ""))}", '
                     f'fontsize=8];')
    lines.append("}")
    return "\n".join(lines)


# ---------------------------------------------------------------------------
# Self-tests (Stage 0 for the probabilistic layer)
# ---------------------------------------------------------------------------
def _toy_two_hop_kb():
    kb = kbe.KB()
    kb.add_fact("cross_references", ("Art13", "Art6"),
                {"provenance": "Art.13 refers to Art.6", "hallucination_verdict": "ENTAILED",
                 "decoy_certificate": {"W_i": 0.9, "T": 0.4, "alpha": 0.2},
                 "entrapment_certificate": {"FDP_hat": 0.05, "r": 1}})
    kb.add_fact("grants_right", ("Art6", "lawful processing"),
                {"provenance": "Art.6 grants the right to lawful processing",
                 "hallucination_verdict": "ENTAILED",
                 "decoy_certificate": {"W_i": 0.7, "T": 0.4, "alpha": 0.2},
                 "entrapment_certificate": {"FDP_hat": 0.05, "r": 1}})
    kb.add_rule("relevant_right", "relevant_right", (V("A"), V("R")),
                [("cross_references", (V("A"), V("B"))), ("grants_right", (V("B"), V("R")))])
    lw = {("cross_references", ("Art13", "Art6")): 0.9,
          ("grants_right", ("Art6", "lawful processing")): 0.7}
    concl = [("relevant_right", ("Art13", "lawful processing"))]
    return kb, lw, concl


def selftest() -> None:
    # (1) calibration + weight maps
    assert abs(calibrate(0.5) - 0.5) < 1e-12
    assert calibrate(0.0) == EPS and calibrate(1.0) == 1.0 - EPS and calibrate(None) == 0.5
    assert abs(weight_gate_consistent(0.8, 0.2) - 0.8 * 0.8) < 1e-9
    assert abs(weight_identity(0.8) - 0.8) < 1e-9
    assert abs(weight_margin(0.0) - 0.5) < 1e-9 and abs(weight_margin(1.0) - (1.0 - EPS)) < 1e-9
    assert weight_gate_consistent(0.8, 0.2) <= weight_identity(0.8) + 1e-12  # shrinkage <= identity

    # (2) toy 2-hop: marginal == 0.9*0.7 == 0.63 (atom sanitisation incl. space in 'lawful processing')
    kb, lw, concl = _toy_two_hop_kb()
    program, qmap = build_program(kb, lw, concl)
    assert "relevant_right(A,R)" in program and "::cross_references('Art13','Art6')." in program
    wm, eng_w = run_wmc(kb, lw, concl)
    assert eng_w == "fallback_exact_wmc"
    assert abs(wm[("relevant_right", ("Art13", "lawful processing"))] - 0.63) < 1e-9, wm
    if _PROBLOG_OK:
        pm = run_problog(program, qmap)
        assert abs(pm[("relevant_right", ("Art13", "lawful processing"))] - 0.63) < 1e-9, pm
        # (3) fallback-equivalence: WMC == ProbLog within 1e-9
        assert abs(pm[("relevant_right", ("Art13", "lawful processing"))]
                   - wm[("relevant_right", ("Art13", "lawful processing"))]) < 1e-9

    # (4) two-proof noisy-OR sanity: two independent bridges -> 1-(1-0.5)*(1-0.4) = 0.7
    kb2 = kbe.KB()
    # proof 1: a -B1- c ; proof 2: a -B2- c  (two distinct intermediaries, shared conclusion)
    kb2.add_fact("cross_references", ("A", "B1"), {})
    kb2.add_fact("grants_right", ("B1", "C"), {})
    kb2.add_fact("cross_references", ("A", "B2"), {})
    kb2.add_fact("grants_right", ("B2", "C"), {})
    kb2.add_rule("relevant_right", "relevant_right", (V("X"), V("R")),
                 [("cross_references", (V("X"), V("Y"))), ("grants_right", (V("Y"), V("R")))])
    # weights chosen so proof1 prob = 0.5 (sqrt-ish) ... use explicit leaf weights:
    lw2 = {("cross_references", ("A", "B1")): 1.0 - EPS, ("grants_right", ("B1", "C")): 0.5,
           ("cross_references", ("A", "B2")): 1.0 - EPS, ("grants_right", ("B2", "C")): 0.4}
    concl2 = [("relevant_right", ("A", "C"))]
    wm2, _ = run_wmc(kb2, lw2, concl2)
    # proof1 ~ (1-eps)*0.5, proof2 ~ (1-eps)*0.4 ; exact WMC over 4 indep leaves
    # noisy-OR (shared leaf? none shared here) -> 1-(1-0.5')*(1-0.4') with 0.5'=(1-eps)*0.5
    exp = 1.0 - (1.0 - (1.0 - EPS) * 0.5) * (1.0 - (1.0 - EPS) * 0.4)
    assert abs(wm2[("relevant_right", ("A", "C"))] - exp) < 1e-6, (wm2, exp)
    if _PROBLOG_OK:
        program2, qmap2 = build_program(kb2, lw2, concl2)
        pm2 = run_problog(program2, qmap2)
        assert abs(pm2[("relevant_right", ("A", "C"))] - wm2[("relevant_right", ("A", "C"))]) < 1e-9

    # (5) prob trace-graph: leaf prob = weight, root prob = marginal, certs preserved
    proofs = kb.derive_all()
    marg, eng = conclusion_marginals(kb, lw, [(p["atom"][0], p["atom"][1]) for p in proofs])
    g = proof_to_prob_graph(proofs[0], lw, marg)
    root = next(n for n in g["nodes"] if n["kind"] == "derived")
    assert abs(root["prob"] - 0.63) < 1e-9
    leaves = [n for n in g["nodes"] if n["kind"] == "leaf"]
    assert len(leaves) == 2 and all(lf["cert"] and lf["prob"] is not None for lf in leaves)
    dot = prob_graph_to_dot(g, "toy")
    assert dot.startswith("digraph prob_proof {") and "p=0.630" in dot

    print(f"prob_reasoner selftest PASSED (problog_available={_PROBLOG_OK}, engine={eng})")


if __name__ == "__main__":
    selftest()

## Step 1 — Validate the engines (self-tests)

Run both modules' self-tests. `prob_reasoner.selftest()` proves the **ProbLog marginals equal
the exact-WMC marginals to 1e-9**, including:
* the toy 2-hop chain `cross_references(a,b) ∧ grants_right(b,r) ⇒ relevant_right(a,r)` with
  marginal `0.9 × 0.7 = 0.63`,
* a two-proof (shared-conclusion) noisy-OR sanity check,
* a probabilistic trace-graph whose leaf probs equal their weights and whose root prob equals
  the marginal, with certificates preserved.

In [ ]:
if RUN_SELFTESTS:
    kbe.selftest()        # kb_engine (the deterministic engine)
    selftest()            # prob_reasoner (ProbLog == exact-WMC, to 1e-9)
else:
    print("self-tests skipped (RUN_SELFTESTS=False)")

## Step 2 — Reproduce real-document marginals via ProbLog

Each curated document ships with the exact **weighted ProbLog program** that the pipeline
generated for it. We re-evaluate those programs here and confirm the recomputed multi-hop
marginals reproduce the published values — on real **legal / news / regulatory** documents.

* multi-hop documents (`kind="multi_hop"`) are evaluated with ProbLog;
* the depth-1 `kind="admission"` document has a single probabilistic leaf, so its marginal is
  just the leaf's calibrated weight (no bridge rule fires).

In [ ]:
n_docs = min(MAX_TRACE_DOCS, len(data["prob_trace_graphs"]))
print(f"Re-evaluating {n_docs} curated document(s)\n")
rows = []
for t in data["prob_trace_graphs"][:n_docs]:
    stored = {(m["conclusion"][0], tuple(m["conclusion"][1])): m["marginal"] for m in t["marginals"]}
    if t["kind"] == "admission":
        recomputed, engine = stored, t["engine"]                 # leaf-weight marginal
    else:
        # rebuild the query map from the stored conclusions, then evaluate with ProbLog
        qmap = {_canon(_render_atom(p, list(a), is_rule=False)): (p, a) for (p, a) in stored.keys()}
        recomputed, engine = run_problog(t["program"], qmap), "problog"
    for k, v in stored.items():
        got = recomputed.get(k)
        ok = (got is not None) and abs(got - v) < 1e-6
        rows.append((t["doc_id"], t["genre"], engine, k, v, got, ok))

print(f"{'doc_id':16s} {'genre':11s} {'engine':9s} {'stored':>9s} {'recomp':>9s}  ok")
print("-" * 64)
for doc, genre, eng, k, v, got, ok in rows:
    print(f"{doc:16s} {genre:11s} {eng:9s} {v:9.4f} {got:9.4f}  {'PASS' if ok else 'FAIL'}")
    print(f"      -> {k[0]}({', '.join(k[1])})"[:96])
print(f"\nAll {len(rows)} stored marginals reproduced: {all(r[6] for r in rows)}")

## Step 3 — Certificate → weight sensitivity

The same multi-hop proofs, re-weighted under the three certificate→weight maps. The default
**gate-consistent shrinkage** `w_i = (1 - alpha_hat) * calibrate(Z_i)` is the most conservative:
its marginal is ≤ the identity (no-shrinkage) marginal **everywhere**, because shrinkage only
pulls leaf weights down.

In [ ]:
sens = data["sensitivity_shrinkage_vs_margin"][:MAX_SENSITIVITY_ROWS]
print(f"{'shrinkage':>10s} {'margin':>8s} {'identity':>9s}   conclusion")
print("-" * 86)
shrink_le_identity = True
for s in sens:
    g = s["marginal_gate_consistent_shrinkage"]
    m = s["marginal_per_pair_margin"]
    i = s["marginal_identity_no_shrinkage"]
    shrink_le_identity &= (g <= i + 1e-9)
    concl = f"{s['conclusion'][0]}({', '.join(s['conclusion'][1])})"
    print(f"{g:10.4f} {m:8.4f} {i:9.4f}   {concl[:60]}")
print(f"\nDEFAULT (gate-consistent shrinkage) <= identity everywhere: {shrink_le_identity}")
print("default map:", data["meta"]["weight_map"]["gate_consistent_shrinkage"])
print("alpha_hat by genre:", data["meta"]["alpha_hat_by_genre"])

## Step 4 — Honest finalized reporting

The headline hallucination-reduction metrics, stated plainly:

* **(i) atomic** hallucinated-fact reduction: raw ≈ 0.245 → gate ≈ 0.18 (~25% relative), but
  **directional, not significant** at n=24 (0 / 40 grid and 0 / 10 pooled cells have a
  raw−gate difference CI excluding zero).
* **(ii) multi-hop** conclusion corruption (raw-KB → gate-KB at α=0.5): 0.52 → 0.25, from a
  single contributing genre (regulatory), with a wide CI.
* **self-report regime = conservative**: `decoy_fdr_hat ≥ realized` in **all 40 cells**
  (0 anti-conservative) — the *opposite* of the CLUTRR multi-hop anti-conservative regime
  (`decoy_fdr_hat = 0.5 < realized = 1.0`).

In [ ]:
hr = data["honest_reporting"]
arp, mhc, srr = hr["atomic_reduction_pooled"], hr["multihop_corruption"], hr["self_report_analysis"]

print("== (i) Atomic hallucinated-fact reduction (pooled, alpha=%.2g) ==" % arp["headline_alpha"])
print("   raw ~0.245 -> gate ~0.18   (~25% relative, DIRECTIONAL not significant)")
print("   CI-separated cells: %d/%d grid, %d/%d pooled" % (
    arp["cells_ci_separated_allgrid"], arp["n_allgrid_cells"],
    arp["cells_ci_separated_pooled"], arp["n_pooled_cells"]))

print("\n== (ii) Multi-hop conclusion corruption (RAW-KB -> GATE-KB, alpha=0.5) ==")
draw = mhc["ci"]["raw_corrupted_rate"]; dgate = mhc["ci"]["gate_a0.5_corrupted_rate"]
ddelta = mhc["ci"]["delta_raw_minus_gate"]
print("   raw  = %.3f  CI=[%.3f, %.3f]   (%d derived, %d corrupt)" % (
    draw["point"], draw["ci"][0], draw["ci"][1], mhc["pooled"]["raw"]["derived"], mhc["pooled"]["raw"]["corrupt"]))
print("   gate = %.3f  CI=[%.3f, %.3f]   (%d derived, %d corrupt)" % (
    dgate["point"], dgate["ci"][0], dgate["ci"][1],
    mhc["pooled"]["gate_a0.5"]["derived"], mhc["pooled"]["gate_a0.5"]["corrupt"]))
print("   delta= %.3f  CI=[%.3f, %.3f]  excludes 0: %s   single-genre origin: %s" % (
    ddelta["point"], ddelta["ci"][0], ddelta["ci"][1], ddelta["ci_excludes_zero"], mhc["single_genre_origin"]))

print("\n== self-report regime: %s ==" % srr["regime"])
print("   anti-conservative cells: %d / %d   (tau=%.2g)" % (
    srr["anti_conservative_cells"], srr["n_cells"], srr["tau"]))
print("   CLUTRR contrast: decoy_fdr_hat=%.2g vs realized=%.2g (anti-conservative)" % (
    srr["clutrr_contrast"]["clutrr_multihop_decoy_fdr_hat"], srr["clutrr_contrast"]["clutrr_multihop_realized"]))
print("   ->", srr["interpretation"])

## Step 5 — Figures

* **F1** — pooled atomic hallucination rate, raw vs decoy-gate (the certified α=0.5 cell),
  for both elicitations, with bootstrap CIs.
* **F2** — multi-hop conclusion corruption, raw-KB vs gate-KB, plus per-genre derived/corrupt
  counts (the reduction comes entirely from the regulatory genre).
* **F3** — `decoy_fdr_hat` vs realized FDR per grid cell; points on/above the `y=x` line mean
  the certificate is a **conservative** (upper) bound — contrast the CLUTRR multi-hop point,
  which is anti-conservative (below the line).

In [ ]:
F = data["figures"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

# --- F1: pooled atomic raw vs gate at the certified alpha=0.5 cell, both elicitations ---
ax = axes[0]
f1 = F["F1"]; ai = f1["alpha_grid"].index(0.5)
elics = list(f1["by_elic"].keys()); x = range(len(elics)); w = 0.36
raw = [f1["by_elic"][e]["raw"][ai] for e in elics]
gate = [f1["by_elic"][e]["gate"][ai] for e in elics]
raw_err = [[raw[j] - f1["by_elic"][e]["raw_ci"][ai][0] for j, e in enumerate(elics)],
           [f1["by_elic"][e]["raw_ci"][ai][1] - raw[j] for j, e in enumerate(elics)]]
gate_err = [[gate[j] - f1["by_elic"][e]["gate_ci"][ai][0] for j, e in enumerate(elics)],
            [f1["by_elic"][e]["gate_ci"][ai][1] - gate[j] for j, e in enumerate(elics)]]
ax.bar([i - w/2 for i in x], raw, w, yerr=raw_err, capsize=4, label="raw LLM", color="lightsalmon")
ax.bar([i + w/2 for i in x], gate, w, yerr=gate_err, capsize=4, label="decoy-gate", color="palegreen")
ax.set_xticks(list(x)); ax.set_xticklabels(elics)
ax.set_ylabel("hallucinated-fact rate"); ax.set_title("F1: pooled atomic (alpha=0.5)")
ax.legend(fontsize=8)

# --- F2: multi-hop corruption raw vs gate + per-genre counts ---
ax = axes[1]
f2 = F["F2"]
pts = [f2["raw_point"], f2["gate_point"]]
errs = [[pts[0] - f2["raw_ci"][0], pts[1] - f2["gate_ci"][0]],
        [f2["raw_ci"][1] - pts[0], f2["gate_ci"][1] - pts[1]]]
ax.bar(["raw-KB", "gate-KB\n(alpha=0.5)"], pts, yerr=errs, capsize=5,
       color=["lightsalmon", "palegreen"])
for i, p in enumerate(pts):
    ax.text(i, p + 0.02, f"{p:.2f}", ha="center", fontsize=9)
ax.set_ylabel("multi-hop corruption rate")
ax.set_title("F2: multi-hop corruption (origin: %s)" % f2["single_genre_origin"])
ax.set_ylim(0, 1)

# --- F3: decoy_fdr_hat vs realized scatter (conservative = on/above y=x) ---
ax = axes[2]
f3 = F["F3"]
gx = [p["decoy_fdr_hat"] for p in f3["points"]]
gy = [p["realized_fdr"] for p in f3["points"]]
ax.scatter(gx, gy, c="steelblue", s=28, alpha=0.7, label="anchor grid cells")
cp = f3["clutrr_point"]
ax.scatter([cp["decoy_fdr_hat"]], [cp["realized_fdr"]], c="crimson", s=90, marker="*",
           label=cp["label"] + " (anti-cons.)")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="y = x")
ax.fill_between([0, 1], [0, 1], [1, 1], color="palegreen", alpha=0.2)
ax.set_xlabel("decoy_fdr_hat (self-report)"); ax.set_ylabel("realized FDR")
ax.set_title("F3: self-report regime = %s" % f3["regime"])
ax.set_xlim(-0.03, 1.03); ax.set_ylim(-0.03, 1.03); ax.legend(fontsize=7)

plt.tight_layout(); plt.show()

### F4 — Probabilistic trace-graph

A regulatory multi-hop conclusion, rendered with `networkx`. **Leaf** nodes (rounded boxes)
show the gate-consistent shrinkage weight `w_i` and the gate verdict — **salmon = HALLUCINATED**,
**green = ENTAILED** — together with the decoy `(W_i, T, α)` certificate. The **derived** node
(blue) shows the ProbLog/WMC **marginal**: the hallucinated leaf (its knockoff statistic
`W_i < T`) is correctly down-weighted, so the conclusion's marginal is low.

In [ ]:
# locate the requested document's trace-graph (fall back to the first multi_hop one)
trace = next((t for t in data["prob_trace_graphs"] if t["doc_id"] == F4_DOC_ID), None)
if trace is None or trace["kind"] == "admission":
    trace = next(t for t in data["prob_trace_graphs"] if t["kind"] == "multi_hop")
g = trace["graphs"][0]   # first proof of this document

def short(label, n=42):
    return label if len(label) <= n else label[:n - 1] + "…"

G = nx.DiGraph()
for nd in g["nodes"]:
    G.add_node(nd["id"])
for e in g["edges"]:
    G.add_edge(e["src"], e["dst"])

# layered top-down layout by BFS depth from the root (node 0)
depth = {0: 0}
for e in g["edges"]:
    depth[e["dst"]] = depth.get(e["src"], 0) + 1
by_level = {}
for nd in g["nodes"]:
    by_level.setdefault(depth.get(nd["id"], 0), []).append(nd["id"])
pos, labels, colors = {}, {}, []
for nd in g["nodes"]:
    lv = depth.get(nd["id"], 0)
    row = by_level[lv]
    pos[nd["id"]] = (row.index(nd["id"]) - (len(row) - 1) / 2.0, -lv)
    if nd["kind"] == "derived":
        labels[nd["id"]] = f"{short(nd['label'])}\nrule: {nd.get('rule')}\np = {nd.get('prob'):.3f}"
        colors.append("lightblue")
    else:
        cert = nd.get("cert") or {}; dc = cert.get("decoy_certificate") or {}
        hv = cert.get("hallucination_verdict", "?")
        labels[nd["id"]] = (f"{short(nd['label'])}\n[{hv}]  w = {nd.get('prob'):.3f}\n"
                            f"W={dc.get('W_i')} T={dc.get('T')} a={dc.get('alpha')}")
        colors.append("lightsalmon" if hv == "HALLUCINATED" else "palegreen")

fig, ax = plt.subplots(figsize=(13, 6))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowsize=18,
                       node_size=16000, min_target_margin=28)
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=colors, node_size=16000,
                       node_shape="s", edgecolors="black")
nx.draw_networkx_labels(G, pos, labels, ax=ax, font_size=8)
ax.set_title(f"F4: probabilistic trace-graph  [{trace['doc_id']} / {trace['genre']} / engine={trace['engine']}]")
ax.axis("off"); plt.tight_layout(); plt.show()

print("conclusion marginal:", trace["marginals"][0]["marginal"])
print("(rule:", g["nodes"][0].get("rule"), "| leaves down-weighted by their decoy certificates)")